# Gikomba Market Stall Performance Analysis

**Analyst:** Dennis Thanji Gitau | **Cohort:** PLP Data Science

Wanjiku runs a fresh-produce stall at Gikomba Market and has recorded 14 days of sales plus a rough stock record. This notebook answers three questions: how is the business doing, is it growing, and what should she restock and focus on. Core Python only (no pandas or NumPy).

## Data

The dataset exactly as provided.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics

I use `sum()`, `max()`, `min()` and `len()` for the headline numbers. `list.index()` gives the position of the best and worst days. Positions 0-6 are week 1, so the week number is `position // 7 + 1` and the day name comes from the parallel `days` list.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_idx = daily_sales.index(highest)
worst_idx = daily_sales.index(lowest)
best_day, best_week = days[best_idx], best_idx // 7 + 1
worst_day, worst_week = days[worst_idx], worst_idx // 7 + 1

print(f"Total sales:   KES {total_sales:,}")
print(f"Average daily: KES {average_sales:,.2f}")
print(f"Highest:       KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest:        KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales:   KES 46,450
Average daily: KES 3,317.86
Highest:       KES 5,600 (Sat, week 2)
Lowest:        KES 1,950 (Tue, week 1)


## Task 2: Week-on-week growth

`week_totals()` slices the list into the two weeks and returns both totals as a tuple, which I unpack into two variables. Growth is (week 2 - week 1) / week 1 x 100.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a 14-day list of daily sales.

    Args:
        sales: list of 14 daily sales figures; the first 7 are week 1.

    Returns:
        A tuple (week1_total, week2_total).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1, week2 = week_totals(daily_sales)
growth_pct = (week2 - week1) / week1 * 100

print(f"Week 1 total: KES {week1:,}")
print(f"Week 2 total: KES {week2:,}")
print(f"Week-on-week growth: {growth_pct:+.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: +10.7%


## Task 3: Classifying days

`classify_day()` compares a day's sales to the 14-day average: more than 20% above is Strong, more than 20% below is Weak, anything else is Normal. I apply it to every day and count the results with the dictionary counting pattern (`dict.get(key, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Args:
        amount: sales for the day (KES).
        average: average daily sales (KES).

    Returns:
        "Strong" if amount is more than 20% above average,
        "Weak" if more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    if amount < average * 0.8:
        return "Weak"
    return "Normal"


day_classes = [classify_day(amount, average_sales) for amount in daily_sales]

class_counts = {}
for label in day_classes:
    class_counts[label] = class_counts.get(label, 0) + 1

for label in ["Strong", "Normal", "Weak"]:
    print(f"{label:<7}: {class_counts.get(label, 0)} days")

Strong : 4 days
Normal : 4 days
Weak   : 6 days


## Task 4: Weekdays vs weekends

Each week's first five entries are Mon-Fri and the last two are Sat-Sun. I collect them with a loop over `zip(days, daily_sales)` and compare the averages.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends are {weekend_uplift_pct:.1f}% higher than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends are 82.5% higher than weekdays


## Task 5: Restocking and margins

I loop over `stock.items()`. `products_to_reorder()` returns products where quantity is at or below the reorder level. Gross margin is (price - cost) / price x 100 for each product.

In [6]:
def products_to_reorder(stock_dict):
    """Return a list of product names at or below their reorder level.

    Args:
        stock_dict: dict of product -> {"qty", "reorder", "cost", "price"}.

    Returns:
        List of product names where qty <= reorder.
    """
    return [name for name, info in stock_dict.items() if info["qty"] <= info["reorder"]]


reorder_list = products_to_reorder(stock)

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Reorder now:", ", ".join(reorder_list))
print()
for name, margin in margins.items():
    print(f"{name:<10} gross margin: {margin:.1f}%")

Reorder now: Tomatoes, Sukuma

Tomatoes   gross margin: 33.3%
Onions     gross margin: 30.0%
Sukuma     gross margin: 40.0%
Potatoes   gross margin: 33.3%
Avocado    gross margin: 42.9%


## Task 6: Stall Performance Report

All figures come from the earlier cells and are printed with f-strings. Money uses the `,` thousands separator and columns are aligned with width specifiers.

In [7]:
line = "=" * 46
print(line)
print("STALL PERFORMANCE REPORT - WANJIKU, GIKOMBA".center(46))
print(line)

print("\nSALES OVERVIEW (14 days)")
print(f"  {'Total sales':<22}KES {total_sales:>12,}")
print(f"  {'Average daily sales':<22}KES {average_sales:>12,.2f}")
print(f"  {'Best day':<22}KES {highest:>12,}  ({best_day}, wk {best_week})")
print(f"  {'Worst day':<22}KES {lowest:>12,}  ({worst_day}, wk {worst_week})")

print("\nGROWTH")
print(f"  {'Week 1 total':<22}KES {week1:>12,}")
print(f"  {'Week 2 total':<22}KES {week2:>12,}")
print(f"  {'Week-on-week growth':<22}{growth_pct:>+16.1f}%")

print("\nDAY QUALITY")
for label in ["Strong", "Normal", "Weak"]:
    print(f"  {label:<22}{class_counts.get(label, 0):>16} days")

print("\nWEEKDAY vs WEEKEND (average per day)")
print(f"  {'Weekday (Mon-Fri)':<22}KES {weekday_avg:>12,.2f}")
print(f"  {'Weekend (Sat-Sun)':<22}KES {weekend_avg:>12,.2f}")
print(f"  {'Weekend uplift':<22}{weekend_uplift_pct:>+16.1f}%")

print("\nSTOCK & MARGINS")
print(f"  {'Product':<10}{'Qty':>5}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in reorder_list else "OK"
    print(f"  {name:<10}{info['qty']:>5}{info['reorder']:>9}{margins[name]:>8.1f}%  {status}")
print(line)

 STALL PERFORMANCE REPORT - WANJIKU, GIKOMBA  

SALES OVERVIEW (14 days)
  Total sales           KES       46,450
  Average daily sales   KES     3,317.86
  Best day              KES        5,600  (Sat, wk 2)
  Worst day             KES        1,950  (Tue, wk 1)

GROWTH
  Week 1 total          KES       22,050
  Week 2 total          KES       24,400
  Week-on-week growth              +10.7%

DAY QUALITY
  Strong                               4 days
  Normal                               4 days
  Weak                                 6 days

WEEKDAY vs WEEKEND (average per day)
  Weekday (Mon-Fri)     KES     2,685.00
  Weekend (Sat-Sun)     KES     4,900.00
  Weekend uplift                   +82.5%

STOCK & MARGINS
  Product     Qty  Reorder   Margin  Status
  Tomatoes     18       25    33.3%  REORDER
  Onions       40       20    30.0%  OK
  Sukuma        9       15    40.0%  REORDER
  Potatoes     55       30    33.3%  OK
  Avocado      12       10    42.9%  OK


## Task 7: Recommendations for Wanjiku

Your business is growing: week 2 sales were KES 24,400 against KES 22,050 in week 1, a rise of 10.7%. Weekends are your money days, averaging KES 4,900 against KES 2,685 on weekdays (82.5% higher), and all 4 Strong days were Saturdays or Sundays, so make sure you are fully stocked before Friday evening. Restock Tomatoes (18 kg against a reorder level of 25 kg) and Sukuma (9 kg against 15 kg) now, and push Avocado (42.9% margin) and Sukuma (40%) because they earn the most per kilo, while watching Avocado, which is only 2 kg above its reorder level. Do not buy more Onions (40 kg in stock, 30% margin) or Potatoes (55 kg in stock) for now. Since 6 of your 14 days were Weak and every one was a weekday, try weekday offers on Onions and Potatoes to move that stock.